In [11]:
class POS:
    def __init__(self):
        self.pos=['PRON','VERB','NOUN','ADV']

    def train(self,sentences,tags):
        self.initial_proba={pos:0 for pos in self.pos}
        self.all_words=[]
        
        # creating inital probability
        for i in range(len(sentences)):
            self.initial_proba[tags[i][0]]+=1/len(sentences)
            self.all_words=list(set(self.all_words+sentences[i]))
        
        # creating emmision probability
        
        self.emmision_probab={pos:{word:0 for word in self.all_words} for pos in self.pos}
        for sentence_ind in range(len(sentences)):
            for word_ind in range (len(sentences[sentence_ind])):
                word=sentences[sentence_ind][word_ind]
                pos=tags[sentence_ind][word_ind]
                self.emmision_probab[pos][word]+=1
        
        for pos in self.pos:
            count_non_zero=0
            for word in self.all_words:
                if self.emmision_probab[pos][word]!=0:
                    count_non_zero+=self.emmision_probab[pos][word]
                    
            for word in self.all_words:
                self.emmision_probab[pos][word]/=count_non_zero
            
        # calculating Transition probability
        self.transition_probab={pos:{pos:0 for pos in self.pos} for pos in self.pos}

        for i in range(len(tags)):
            for j in range(len(tags[i])-1):
                self.transition_probab[tags[i][j]][tags[i][j+1]]+=1

        for pos1 in self.pos:
            count_non_zero=0
            for pos2 in self.pos:
                if self.transition_probab[pos1][pos2]!=0:
                    count_non_zero+=self.transition_probab[pos1][pos2]
                    
            for pos2 in self.pos:
                self.transition_probab[pos1][pos2]/=count_non_zero
        
    def predict(self,sentence):
        dp=[{} for i in range(len(sentence))]
        best=[{} for i in range(len(sentence))]
        for pos in self.pos:
            dp[0][pos]=self.initial_proba[pos]*self.emmision_probab[pos][sentence[0]]
            best[0][pos]=None

        for t in range(1,len(sentence)):
            word=sentence[t]
            for curr_pos in self.pos:
                max_probab=0
                best_prev=None
                emission=self.emmision_probab[curr_pos][word]
                for prev_pos in self.pos:
                    trans=self.transition_probab[prev_pos][curr_pos]
                    prob=dp[t-1][prev_pos]*trans*emission
                    if prob>max_probab:
                        max_probab=prob
                        best_prev=prev_pos

                dp[t][curr_pos]=max_probab
                best[t][curr_pos]=best_prev


        last_tag=max(dp[len(sentence)-1],key=lambda x:dp[len(sentence)-1][x])
        tags=[last_tag]
        for t in range(len(sentence)-1,0,-1):
            last_tag=best[t][last_tag]
            tags.append(last_tag)
        return tags[::-1]

In [12]:
ps=POS()
sentences=[
    ['I','eat','fish','daily'],
    ['She','eats','cake','now'],
    ['They','fish','well','now']
]

tags=[
    ['PRON','VERB','NOUN','ADV'],
    ['PRON','VERB','NOUN','ADV'],
    ['PRON','VERB','ADV','ADV']
]
ps.train(sentences,tags)
ps.predict(['They','fish','now','well'])

['PRON', 'VERB', 'ADV', 'ADV']